In [3]:
from pathlib import Path
import sys
import os

%load_ext autoreload
%autoreload 2

dir = Path().resolve().parents[1]

if dir not in sys.path:
    print("directory path is not in the system path")
    sys.path.append(str(dir))
    print("adding directory...")
else:
    print("Directory already exists in the system path")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
directory path is not in the system path
adding directory...


In [ ]:
from nn import Unet1D, Returns, RMSELoss
from scripts import train
from utils import log_transform
import torch
from torch.utils.data import DataLoader, random_split
import yfinance as yf
import pandas as pd
import numpy as np
import math

In [100]:
ticker = "^GSPC"
start_interval = "2016-12-01"
end_interval = "2026-01-01"
interval = "1d"

raw_snp500 = torch.tensor(yf.Ticker(ticker).history(start=start_interval, end=end_interval, interval=interval)["Close"].to_numpy())

In [103]:
split_idx = len(raw_snp500) - math.ceil(len(raw_snp500) * 0.2)
train_raw_snp500, test_raw_snp500 = raw_snp500[:split_idx], raw_snp500[split_idx:]

window_size = 32

train_data = Returns(
  raw_returns=train_raw_snp500,
  window_size=window_size,
  transform=log_transform
)
test_data = Returns(
  raw_returns=test_raw_snp500,
  window_size=window_size,
  transform=log_transform
)

len(train_data), len(test_data)

(1794, 425)

In [104]:
train_dataloader = DataLoader(train_data, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=True)

next(iter(train_dataloader)).size()

torch.Size([32, 1, 32])

In [105]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.cuda.manual_seed(42)
device

'cpu'

In [107]:
encoder_in_channels = []
encoder_out_channels = []
decoder_in_channels = []
decoder_out_channels = []
attn_res = 16
n_res_block = 2
T = 1000

# model = Unet1D(
#   attn_res=attn_res,
#   n_res_block=n_res_block,
#   encoder_in_channels=encoder_in_channels,
#   encoder_out_channels=encoder_out_channels,
#   decoder_in_channels=decoder_in_channels,
#   decoder_out_channels=decoder_out_channels,
#   T=T
# )

# loss_fn = RMSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)